# Analysis

**Hypothesis**: Within ventricular cardiomyocytes, spatial gradients of developmental transcription factor expression form coherent, spatially localized maturation states that are associated with differences in local cellular purity (segmentation confidence) and are not captured by existing Leiden clusters or coarse cell-type labels.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# Set up visualization defaults for better plots
sc.settings.verbosity = 3
sc.settings.figsize = (8, 8)
sc.settings.dpi = 100
sc.settings.facecolor = 'white'
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['savefig.dpi'] = 150
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.2)

# Load data
print("Loading data...")
adata = sc.read_h5ad("/home/mingqiam/TissueAgent/demo/data/dataset_farah_heart_merfish.h5ad")
print(f"Data loaded: {adata.shape[0]} cells and {adata.shape[1]} genes")


Loading data...


Data loaded: 228635 cells and 238 genes


# Analysis Plan

**Hypothesis**: Within ventricular cardiomyocytes, spatial gradients of developmental transcription factor expression form coherent, spatially localized maturation states that are associated with differences in local cellular purity (segmentation confidence) and are not captured by existing Leiden clusters or coarse cell-type labels.

## Steps:
- Confirm and summarize the distribution of ventricular cardiomyocyte subtypes and Purity values across samples and Leiden clusters, and identify key developmental transcription factor and signaling genes present in the MERFISH panel that will serve as a maturation/signaling signature, storing this gene set in adata.uns for reuse.
- Within ventricular cardiomyocyte populations only, compute per-cell standardized TF/signaling scores (e.g. z-scored gene expressions averaged into a signature), quantify spatial autocorrelation (Moran’s I) of these scores within each sample and subtype using spatial coordinates, and assess association of the scores with Purity using correlation and regression models.
- Define spatial neighborhoods for ventricular cardiomyocytes using a k-nearest-neighbors graph on spatial coordinates, compute neighborhood-averaged TF/signaling scores, and test whether local neighborhood scores differ significantly between high- and low-Purity cells within each subtype and sample, reporting effect sizes and p-values.
- Cluster ventricular cardiomyocytes based on TF/signaling gene expression alone (e.g. PCA + Leiden on this restricted gene set), and evaluate how these TF/signaling-defined states map onto spatial location, Purity distributions, and existing Leiden clusters/Populations via contingency tables and enrichment tests.
- Extend TF/signaling scoring to selected non-cardiomyocyte populations (e.g., fibroblasts, endothelial cells) and, within shared spatial neighborhoods, test for correlations between local cardiomyocyte TF/signaling scores and neighboring non-cardiomyocyte TF/signaling scores to assess coordinated spatial maturation across cell types.
- Summarize statistically significant spatial gradients, TF-defined maturation states, and cross-cell-type associations, reporting effect sizes and p-values for key tests (Moran’s I, correlations, regressions, and group comparisons) in text form without figures.


## This code expands the initial overview step by summarizing ventricular cardiomyocyte populations, Purity, and their distribution across samples and Leiden clusters, while identifying developmental TF/signaling genes and storing them in adata.uns for downstream reuse. It prepares the dataset for spatial and TF-signature-based analyses by quantifying basic expression statistics of these candidate genes within ventricular cardiomyocytes.

In [ ]:

import numpy as np
import pandas as pd
import scanpy as sc

# 1) Basic overview of adata and relevant metadata columns
print("AnnData shape (cells x genes):", adata.shape)
print("\n.obs columns:", list(adata.obs.columns))
print("\n.obsm keys:", list(adata.obsm.keys()))

# 2) Summarize ventricular cardiomyocyte-related populations
if 'Populations' not in adata.obs.columns:
    raise ValueError("Expected 'Populations' column in adata.obs but did not find it.")

vc_labels = [c for c in adata.obs['Populations'].unique().tolist() if isinstance(c, str) and c.startswith('vCM')]
print("\nVentricular CM-related Populations:")
for lab in vc_labels:
    n_cells = (adata.obs['Populations'] == lab).sum()
    print(f"  {lab}: {n_cells} cells")
if len(vc_labels) == 0:
    print("  No Populations entries starting with 'vCM' were found.")

# 3) Purity distribution overall, by sample, by Leiden, and within ventricular CM populations
if 'Purity' not in adata.obs.columns:
    raise ValueError("Expected 'Purity' column in adata.obs but did not find it.")

print("\nOverall Purity summary:")
print(adata.obs['Purity'].describe())

print("\nPurity summary by Sample_ID:")
if 'Sample_ID' in adata.obs.columns:
    print(adata.obs.groupby('Sample_ID')['Purity'].describe())
else:
    print("Sample_ID column not found; skipping per-sample Purity summary.")

print("\nPurity summary by Leiden cluster (all cells):")
if 'leiden' in adata.obs.columns:
    print(adata.obs.groupby('leiden')['Purity'].describe())
else:
    print("leiden column not found; skipping Purity by Leiden summary.")

print("\nPurity summary within ventricular CM Populations:")
vc_mask = adata.obs['Populations'].isin(vc_labels)
if vc_mask.sum() == 0:
    print("No ventricular cardiomyocyte cells found based on 'vCM' prefix.")
else:
    vc_df = adata.obs.loc[vc_mask, ['Populations', 'Purity']]
    print(vc_df.groupby('Populations')['Purity'].describe())

    # Cross-tabulate ventricular CM Populations with Leiden clusters
    if 'leiden' in adata.obs.columns:
        print("\nContingency table of ventricular CM Populations vs Leiden clusters:")
        ct = pd.crosstab(adata.obs.loc[vc_mask, 'Populations'], adata.obs.loc[vc_mask, 'leiden'])
        print(ct)
    else:
        print("\nleiden column not found; skipping Populations vs Leiden contingency table.")

    # Optional: summarize Purity by Leiden within vCMs
    if 'leiden' in adata.obs.columns:
        print("\nPurity summary by Leiden within ventricular CM cells:")
        print(adata.obs.loc[vc_mask].groupby('leiden')['Purity'].describe())

# 4) Identify candidate developmental TF / signaling genes present in the panel
candidate_keywords = ['TBX', 'NKX', 'GATA', 'HAND', 'HEY', 'BMP', 'FGF', 'WNT', 'NOTCH', 'HES', 'SOX', 'MEF2', 'ISL', 'ID', 'PDGF']
var_names = pd.Index(adata.var_names.astype(str))

candidate_genes = []
for g in var_names:
    upper_g = g.upper()
    if any(kw in upper_g for kw in candidate_keywords):
        candidate_genes.append(g)

candidate_genes = sorted(set(candidate_genes))

print("\nCandidate developmental TF/signaling genes found in panel (based on name keywords):")
print(f"  Total candidate genes: {len(candidate_genes)}")
if len(candidate_genes) == 0:
    print("  None detected with the given keyword heuristics.")
else:
    for g in candidate_genes:
        print("  ", g)

# Store candidate gene list in adata.uns for reuse in downstream steps
adata.uns['vc_tf_signaling_genes'] = candidate_genes

# 5) Basic expression summary of candidate genes within ventricular cardiomyocytes
if len(candidate_genes) > 0 and vc_mask.sum() > 0:
    sub = adata[vc_mask, candidate_genes]
    # Convert to dense for summary statistics (panel is small)
    X = sub.X.A if hasattr(sub.X, 'A') else sub.X
    # Ensure X is 2D and matches number of genes
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    # Ensure the number of columns matches the number of genes
    if X.shape[1] != len(sub.var_names):
        X = X.reshape(X.shape[0], len(sub.var_names))
    expr_df = pd.DataFrame(X, columns=list(sub.var_names))
    print("\nVentricular CM cells: ", vc_mask.sum())
    print("Candidate TF/signaling genes: ", len(candidate_genes))
    print("\nPer-gene expression summary within ventricular cardiomyocytes (mean, std, fraction > 0):")
    for g in sub.var_names:
        vals = expr_df[g].values
        frac_pos = np.mean(vals > 0)
        print(f"  {g}: mean={vals.mean():.3f}, std={vals.std(ddof=1):.3f}, frac_expressing={frac_pos:.3f}")
else:
    print("\nSkipping candidate gene expression summaries (no candidate genes or no ventricular CM cells).")


AnnData shape (cells x genes): (228635, 238)

.obs columns: ['Sample_ID', 'Batch', 'UMI Count', 'leiden', 'Complexity', 'Populations', 'Purity']

.obsm keys: ['X_umap', 'spatial']

Ventricular CM-related Populations:
  vCM-LV-AV: 7348 cells
  vCM-RV-AV: 5845 cells
  vCM-His-Purkinje: 5429 cells
  vCM-RV-Compact: 9488 cells
  vCM-LV-Compact: 30380 cells
  vCM-Proliferating: 17584 cells
  vCM-RV-Trabecular: 8052 cells
  vCM-LV-Trabecular: 16511 cells

Overall Purity summary:
count    228635.000000
mean          0.502520
std           0.152212
min           0.135338
25%           0.390805
50%           0.490826
75%           0.599291
max           1.000000
Name: Purity, dtype: float64

Purity summary by Sample_ID:
             count      mean       std       min       25%       50%  \
Sample_ID                                                              
R77_4C4    72962.0  0.497966  0.153822  0.150289  0.382470  0.483553   
R78_4C12   75782.0  0.499514  0.154452  0.135338  0.383592  0.4

ValueError: Shape of passed values is (100637, 1), indices imply (100637, 16)

### Agent Interpretation

Current analysis step failed to run. Try an alternative approach

## Next Steps
Step 1: Robustly verify ventricular cardiomyocyte population definitions and Purity distributions across samples and Leiden clusters, and curate a ventricular developmental TF/signaling gene set into adata.uns['vc_tf_signaling_genes'], avoiding redundant summaries and over-defensive reshaping while ensuring clear failure modes.
Step 2: Within ventricular cardiomyocytes, compute per-cell TF/signaling scores using adata.uns['vc_tf_signaling_genes'], then quantify spatial structure of these scores per sample and vCM subtype by correlating each cell’s score with its spatial kNN-neighborhood mean score (via sc.pp.neighbors on obsm['spatial']) and assessing associations with Purity using Pearson/Spearman correlations and scipy.stats.linregress.
Step 3: Construct spatial kNN neighborhoods among ventricular cardiomyocytes, compute neighborhood-averaged TF/signaling scores, and within each major vCM subtype and sample compare neighborhood scores between high- and low-Purity cells (e.g. per-subtype+sample median split) using effect sizes and Mann–Whitney U or t-tests, with permutation-based p-values where needed.
Step 4: Cluster ventricular cardiomyocytes using only TF/signaling gene expression (PCA + sc.pp.neighbors + sc.tl.leiden on the restricted gene set) and quantitatively compare these TF-defined states to existing Leiden clusters and Populations via contingency tables, chi-squared enrichment tests, and Purity distribution comparisons (Kruskal–Wallis and pairwise Mann–Whitney tests).
Step 5: Extend TF/signaling scoring to key non-cardiomyocyte populations (e.g. ventricular fibroblasts, endothelial cells), and within shared spatial neighborhoods quantify cross-cell-type coordination by correlating local vCM neighborhood TF/signaling scores with neighboring non-CM scores using Pearson/Spearman correlations and simple permutation tests, stratified by Sample_ID.

## This code performs a quality/control overview of an AnnData object to characterize ventricular cardiomyocyte (vCM) populations and their purity, and to curate a panel of developmental transcription factor/signaling genes for downstream analysis. It robustly identifies vCM cells from the Populations metadata, summarizes purity globally and within vCMs (including by sample and cluster where available), heuristically detects TF/signaling genes by name, stores them in `adata.uns`, and computes basic expression statistics for these genes in vCM cells.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

# 1) Basic checks and inventory (without re-loading adata)
print('AnnData shape (cells x genes):', adata.shape)
print('\n.obs columns:', list(adata.obs.columns))
print('\n.obsm keys:', list(adata.obsm.keys()))

# Ensure required obs columns exist
required_obs = ['Populations', 'Purity']
for col in required_obs:
    if col not in adata.obs.columns:
        raise ValueError(f"Expected '{col}' column in adata.obs but did not find it.")

# 2) Define ventricular cardiomyocyte populations robustly
pop_series = adata.obs['Populations'].astype(str)
unique_pops = pop_series.unique().tolist()
print('\nUnique Populations (first 30 shown):', unique_pops[:30])

# Use a robust pattern: any population starting with 'vCM' (case-insensitive)
vc_labels = [p for p in unique_pops if isinstance(p, str) and p.upper().startswith('VCM')]

print('\nVentricular CM-related Populations detected:')
if len(vc_labels) == 0:
    print('  No Populations entries starting with "vCM" (case-insensitive) were found.')
else:
    for lab in sorted(vc_labels):
        n_cells = (pop_series == lab).sum()
        print(f'  {lab}: {n_cells} cells')

# Ventricular CM mask
vc_mask = pop_series.isin(vc_labels)
print(f"\nTotal ventricular CM cells identified: {int(vc_mask.sum())}")

# 3) Purity distribution overall, by Sample_ID (if present), and by Leiden (if present)
print('\nOverall Purity summary:')
print(adata.obs['Purity'].describe())

if 'Sample_ID' in adata.obs.columns:
    print('\nPurity summary by Sample_ID:')
    print(adata.obs.groupby('Sample_ID')['Purity'].describe())
else:
    print('\nSample_ID column not found; skipping per-sample Purity summary.')

if 'leiden' in adata.obs.columns:
    print('\nPurity summary by Leiden cluster (all cells):')
    print(adata.obs.groupby('leiden')['Purity'].describe())
else:
    print('\nleiden column not found; skipping Purity by Leiden summary.')

# 4) Purity within ventricular cardiomyocytes and association with Leiden
if vc_mask.sum() > 0:
    vc_obs = adata.obs.loc[vc_mask, :]
    print('\nPurity summary within ventricular CM Populations:')
    print(vc_obs.groupby('Populations')['Purity'].describe())

    if 'leiden' in vc_obs.columns:
        print('\nContingency table of ventricular CM Populations vs Leiden clusters:')
        ct = pd.crosstab(vc_obs['Populations'], vc_obs['leiden'])
        print(ct)

        print('\nPurity summary by Leiden within ventricular CM cells:')
        print(vc_obs.groupby('leiden')['Purity'].describe())
    else:
        print('\nleiden column not found; skipping Populations vs Leiden contingency table within vCMs.')
else:
    print('\nNo ventricular cardiomyocyte cells found based on Populations pattern; downstream steps will need to be adapted.')

# 5) Robust identification of developmental TF / signaling genes
candidate_keywords = ['TBX', 'NKX', 'GATA', 'HAND', 'HEY', 'BMP', 'FGF', 'WNT', 'NOTCH', 'HES', 'SOX', 'MEF2', 'ISL', 'ID', 'PDGF']

var_names = pd.Index(adata.var_names.astype(str))

candidate_genes = []
for g in var_names:
    upper_g = g.upper()
    if any(kw in upper_g for kw in candidate_keywords):
        candidate_genes.append(g)

candidate_genes = sorted(set(candidate_genes))

print('\nCandidate developmental TF/signaling genes found in panel (based on name keywords):')
print(f'  Total candidate genes: {len(candidate_genes)}')
if len(candidate_genes) == 0:
    print('  None detected with the given keyword heuristics. Downstream TF/signaling analyses will need to be reconsidered.')
else:
    for g in candidate_genes:
        print('  ', g)

# Store candidate gene list in adata.uns for reuse; if none found, store empty list for explicitness
adata.uns['vc_tf_signaling_genes'] = candidate_genes

# 6) Basic expression summary of candidate genes within ventricular cardiomyocytes (if both exist)
if len(candidate_genes) > 0 and vc_mask.sum() > 0:
    sub = adata[vc_mask, candidate_genes].copy()
    X = sub.X

    if X is None:
        print('\nSubsetting returned an AnnData object with X=None; skipping expression summaries.')
    else:
        # Convert sparse to dense if needed
        if hasattr(X, 'A'):
            X = X.A
        X = np.asarray(X)

        # Ensure 2D (cells x genes)
        if X.ndim == 1:
            # 1D vector should be length equal to number of cells; if not, skip safely
            if X.shape[0] == sub.n_obs:
                X = X.reshape(sub.n_obs, 1)
            else:
                print(
                    f"\nWarning: Got 1D expression array of length {X.shape[0]} which does not match number of cells ({sub.n_obs}); "
                    "skipping candidate gene expression summaries to avoid misinterpretation."
                )
                X = None
        elif X.ndim == 0:
            print(
                "\nWarning: Got a scalar expression value for subset; "
                "skipping candidate gene expression summaries."
            )
            X = None

        if X is not None:
            # Final sanity check on gene dimension
            if X.shape[1] != sub.n_vars:
                print(
                    f"\nWarning: Expression matrix second dimension (n_var={X.shape[1]}) "
                    f"does not match number of genes (n_vars={sub.n_vars}); "
                    "skipping candidate gene expression summaries."
                )
            else:
                expr_df = pd.DataFrame(X, columns=list(sub.var_names))

                print(f"\nVentricular CM cells: {int(vc_mask.sum())}")
                print(f"Candidate TF/signaling genes: {len(candidate_genes)}")
                print('\nPer-gene expression summary within ventricular cardiomyocytes (mean, std, fraction > 0):')
                for g in sub.var_names:
                    vals = expr_df[g].values
                    frac_pos = np.mean(vals > 0)
                    mean_val = float(np.mean(vals))
                    std_val = float(np.std(vals, ddof=1)) if vals.size > 1 else 0.0
                    print(f'  {g}: mean={mean_val:.3f}, std={std_val:.3f}, frac_expressing={frac_pos:.3f}')
else:
    print('\nSkipping candidate gene expression summaries (no candidate genes detected or no ventricular CM cells identified).')

print('\nStep 1 summary complete: ventricular CM distribution, Purity patterns, and TF/signaling gene set curated into adata.uns["vc_tf_signaling_genes"].')

AnnData shape (cells x genes): (228635, 238)

.obs columns: ['Sample_ID', 'Batch', 'UMI Count', 'leiden', 'Complexity', 'Populations', 'Purity']

.obsm keys: ['X_umap', 'spatial']

Unique Populations (first 30 shown): ['VIC', 'vCM-LV-AV', 'vCM-RV-AV', 'ncCM-AVC-like', 'EPDC', 'aFibro', 'BEC', 'VSMC', 'vFibro', 'VEC', 'aCM-RA', 'aEndocardial', 'aCM-LA', 'WBC', 'vEndocardial', 'vCM-His-Purkinje', 'LEC', 'vCM-RV-Compact', 'vCM-LV-Compact', 'vCM-Proliferating', 'Pericyte', 'vCM-RV-Trabecular', 'vCM-LV-Trabecular', 'Epicardial', 'Neuronal', 'adFibro', 'ncCM-IFT-like']

Ventricular CM-related Populations detected:
  vCM-His-Purkinje: 5429 cells
  vCM-LV-AV: 7348 cells
  vCM-LV-Compact: 30380 cells
  vCM-LV-Trabecular: 16511 cells
  vCM-Proliferating: 17584 cells
  vCM-RV-AV: 5845 cells
  vCM-RV-Compact: 9488 cells
  vCM-RV-Trabecular: 8052 cells

Total ventricular CM cells identified: 100637

Overall Purity summary:
count    228635.000000
mean          0.502520
std           0.152212
min    

                     count      mean       std       min       25%       50%  \
Populations                                                                    
BEC                    0.0       NaN       NaN       NaN       NaN       NaN   
EPDC                   0.0       NaN       NaN       NaN       NaN       NaN   
Epicardial             0.0       NaN       NaN       NaN       NaN       NaN   
LEC                    0.0       NaN       NaN       NaN       NaN       NaN   
Neuronal               0.0       NaN       NaN       NaN       NaN       NaN   
Pericyte               0.0       NaN       NaN       NaN       NaN       NaN   
VEC                    0.0       NaN       NaN       NaN       NaN       NaN   
VIC                    0.0       NaN       NaN       NaN       NaN       NaN   
VSMC                   0.0       NaN       NaN       NaN       NaN       NaN   
WBC                    0.0       NaN       NaN       NaN       NaN       NaN   
aCM-LA                 0.0       NaN    

### Agent Interpretation

- The ventricular cardiomyocyte (vCM) definition looks robust and nicely aligned with `.obs["Populations"]`: you have ~100k vCM cells across clear anatomical/maturation-related subtypes (LV/RV, AV, compact/trabecular, His‑Purkinje, proliferating). That gives you enough power to look for subtle spatial gradients within and across these subtypes.

- Purity is well-behaved overall (mean ~0.5, reasonably broad spread), and there are clear differences both:
  - Across vCM subtypes (e.g. LV-Compact ~0.47 vs RV-Compact ~0.39 vs LV-Trabecular ~0.48 vs Proliferating ~0.43), and  
  - Across Leiden clusters that are vCM-enriched (e.g. 16, 19 at ~0.50 vs 2, 25 at ~0.40).
  
  This supports the hypothesis that Purity is not random noise and may be biologically or segmentation-structure related. It also means you can legitimately ask whether TF/signaling gradients “track” these Purity differences in space.

- The contingency table shows that existing Leiden clusters are strongly, but not perfectly, aligned to Populations:
  - Some vCM subtypes are almost monophasic in certain Leiden clusters (e.g. vCM-His‑Purkinje ≈ all Leiden 19; vCM-LV-AV heavily in 16; vCM-RV-Compact heavily in 2), while others are spread across several clusters (especially LV-Compact, Proliferating, trabecular populations).
  - That partial mismatch is exactly the regime where your later TF‑only clustering (step 4) can reveal alternative “maturation axes” not captured by the global Leiden solution.

- The TF/signaling gene curation step seems to have worked well in terms of content:
  - You captured a focused set (16 genes) that are very appropriate for developmental and signaling gradients in vCMs: TBX3/5/18, NKX2–5, HAND1/2, ISL1, MEF2C, HEY1/2, BMP2, FGF12, NOTCH1, SOX9, PDGFRA/B.
  - This list is neither trivially huge nor degenerate, so per-cell scores and TF‑based clustering will be interpretable and distinct from the original paper’s broad cell-type analysis.

- However, the warning “Got a scalar expression value for subset; skipping candidate gene expression summaries” is a red flag about the current `adata.X` configuration:
  - A scalar for `sub.X` after subsetting suggests either:
    - `adata.X` might be 1D (e.g. some derived score, not the raw expression matrix), or
    - You are working on a view with an unusual `.X` layout, or
    - The expression has been moved into `.layers` (e.g. `"counts"` or `"lognorm"`) and `X` is something else.
  - This must be fixed before you proceed to steps 2–4, because all your planned TF scoring and TF‑based clustering assume a 2D (cells × genes) expression matrix.

  For next steps, I’d recommend:
  - Explicitly inspect `adata.X.shape`, `adata.layers.keys()`, and `adata.raw`:
    - If counts or log-normalized values live in `adata.layers["counts"]` or similar, use that layer in subsequent scoring (and when subsetting for the TF genes).
    - Alternatively, if the true expression matrix is in `adata.raw.X`, consider using `adata.raw` for all TF/signaling operations.
  - Re-run a minimal check:
    ```python
    sub = adata[vc_mask, adata.uns['vc_tf_signaling_genes']].copy()
    print(sub.X, type(sub.X), getattr(sub.X, 'shape', None))
    ```
    and confirm that you are working on a 2D sparse/dense matrix before computing any scores.

How this informs the hypothesis and future steps:

- Hypothesis relevance:
  - You now have:
    - A well-defined vCM subset,
    - Measurable Purity variation both within and between vCM populations, and
    - A curated developmental TF/signaling gene set.
  - That is sufficient infrastructure to test whether *spatially coherent TF/signaling states* exist in vCMs and whether they relate to Purity independently of Leiden/Populations. The heterogeneity in Purity and in the Populations–Leiden mapping is actually a promising sign that new structure may be discoverable.

- For step 2 (per-cell TF/signaling scores and spatial structure):
  - Once the `X` issue is resolved, I would:
    - Compute a simple TF/signaling score per cell (e.g. mean z‑score across the 16 genes in vCMs, possibly separated into “classic cardiac TFs” vs “signaling receptors/ligands” if you later want more dimensions).
    - Use `sc.pp.neighbors` on `.obsm["spatial"]` *restricted to vCMs* (per sample) to compute a spatial kNN graph and neighborhood-averaged TF scores.
    - Quantify:
      - Correlation of each cell’s TF score with its neighborhood-mean TF score (per sample and per vCM subtype) to assess how spatially smooth/coherent these TF states are.
      - Association of TF scores (or local neighborhood scores) with Purity via Pearson/Spearman and linear models, stratified by sample and subtype. This directly addresses the “local cellular purity” aspect of your hypothesis.
    - Also check whether the TF scores still show structure **within** a single vCM Population–Leiden combination (e.g. only vCM-LV-Compact ∩ Leiden 4), which would demonstrate that TF gradients are not reducible to coarse labels.

- For step 3 (high vs low Purity neighborhood comparison):
  - The substantial spread in Purity within each vCM subtype (e.g. LV-Compact 25–75%: ~0.41–0.53) suggests that a median split within subtype+sample will be meaningful.
  - When constructing spatial kNN neighborhoods, be explicit about:
    - Doing the graph per sample (to avoid cross-section edges),
    - Possibly restricting to vCMs only, then later layering in other cell types.
  - Compare neighborhood-averaged TF scores between high- and low-Purity cells within each subtype+sample, and use effect sizes (e.g. Cliff’s delta or Cohen’s d) alongside Mann–Whitney/t-tests.
  - If you see coherent spatial “patches” where high TF-scores co-occur with either very high or low Purity that cut across Leiden clusters, that would strongly support the core hypothesis.

- For step 4 (TF‑only clustering vs Leiden/Populations):
  - The contingency table already hints that the current Leiden partition is dominated by broad expression patterns, mixing AV, compact, trabecular, proliferating states in complex ways.
  - Using only the 16 TF/signaling genes for PCA + neighbors + Leiden should give a lower-dimensional, developmentally focused manifold. Compare:
    - Whether TF-defined clusters partition cells **within** each existing Population (e.g. split vCM-LV-Compact into multiple TF states), and
    - Whether their Purity distributions differ more than those of the original Leiden clusters.
  - Chi-squared tests on contingency tables (TF-cluster × Populations; TF-cluster × Leiden) will quantify how orthogonal these new states are to the existing labels—this is central to showing that the TF states are “not captured by existing Leiden clusters or coarse cell-type labels.”

- A note to keep the analysis distinct from the paper:
  - The paper likely focused on broad spatial communities and co-localization of cell types. Here you’re explicitly interrogating **fine-scale TF/signaling gradients within a single major cardiomyocyte class** and tying them to segmentation-derived Purity and spatial neighborhood structure. That emphasis on local maturation states + segmentation quality is a distinct analytical angle, so you’re on solid ground in terms of novelty.

In short, the population definition and Purity characterization are in good shape and supportive of the hypothesis. The key technical blocker now is ensuring that you have a valid expression matrix for the TF/signaling genes. Once that is fixed, you can proceed with spatial TF scoring, neighborhood analyses, and TF-based clustering to directly test for spatially coherent maturation states that cut across current Leiden/Population labels and relate to Purity.

## Next Steps
Step 1: Within ventricular cardiomyocytes, compute and store per-cell standardized TF/signaling scores using a robustly selected 2D expression matrix for the curated vc_tf_signaling_genes, keeping this step focused strictly on score generation/QC rather than spatial or Purity modeling.
Step 2: Using the stored vCM TF/signaling scores, construct spatial kNN graphs per Sample_ID based on obsm['spatial'], compute neighborhood-mean TF/signaling scores, and quantify spatial coherence and Purity associations via correlations and linear regression separately (and cleanly) from score creation.
Step 3: Using only the TF/signaling expression matrix within ventricular cardiomyocytes, perform PCA, build a kNN graph, run Leiden clustering to define TF-based maturation states, and compare these TF states to existing Leiden clusters and Populations via contingency tables, chi-squared tests, and Purity distribution tests (Kruskal–Wallis and pairwise Mann–Whitney).
Step 4: Based on the vCM TF-signaling scores and spatial neighborhoods, compare neighborhood-averaged scores between high- and low-Purity vCMs within each major vCM subtype and sample using effect sizes and Mann–Whitney/t-tests, highlighting spatially segregated maturation–Purity couplings not captured by existing labels.

## This code cleanly implements the first analysis step by selecting a valid 2D expression matrix, extracting curated TF/signaling genes within ventricular cardiomyocytes, computing a standardized per-cell TF/signaling score, and storing it in adata.obs while providing basic QC summaries, without yet performing any spatial or Purity association analyses.

In [ ]:

import numpy as np
import pandas as pd
import scanpy as sc

# --- Step 1: Compute and store per-cell vCM TF/signaling scores only ---

# 0) Basic checks
if 'Populations' not in adata.obs.columns:
    raise ValueError("Expected 'Populations' in adata.obs.")
if 'vc_tf_signaling_genes' not in adata.uns:
    raise ValueError("Expected curated TF/signaling gene list in adata.uns['vc_tf_signaling_genes'] from Step 1 overview.")

vc_genes = list(adata.uns['vc_tf_signaling_genes'])
if len(vc_genes) == 0:
    raise ValueError("adata.uns['vc_tf_signaling_genes'] is empty; cannot compute TF/signaling scores.")

# restrict gene list to those present in adata.var_names
vc_genes = [g for g in vc_genes if g in adata.var_names]
if len(vc_genes) == 0:
    raise ValueError("None of the curated TF/signaling genes are present in adata.var_names.")

# 1) Identify ventricular cardiomyocytes (vCMs)
pop = adata.obs['Populations'].astype(str)
vc_labels = [p for p in pop.unique().tolist() if isinstance(p, str) and p.upper().startswith('VCM')]
vc_mask = pop.isin(vc_labels)
if vc_mask.sum() == 0:
    raise ValueError("No ventricular cardiomyocytes found based on Populations starting with 'vCM' (case-insensitive).")

print(f"Total ventricular CM cells: {int(vc_mask.sum())}")
print("vCM Populations:", sorted(set(pop[vc_mask].tolist())))

# 2) Determine which expression matrix to use (robust 2D selection)
expr_source = None

# Helper to check if an array/matrix is usable 2D
def _is_valid_2d(X, n_obs_expected=None):
    if X is None:
        return False
    if hasattr(X, 'A'):
        X = X.A
    X = np.asarray(X)
    if X.ndim != 2:
        return False
    if n_obs_expected is not None and X.shape[0] != n_obs_expected:
        return False
    return True

n_cells = adata.n_obs

# Try adata.X first; if it's 1D or otherwise invalid, try to reshape if length matches
X_try = adata.X
if X_try is not None:
    if hasattr(X_try, 'A'):
        X_try = X_try.A
    X_try = np.asarray(X_try)
    if X_try.ndim == 1 and X_try.shape[0] == n_cells * adata.n_vars:
        X_try = X_try.reshape(n_cells, adata.n_vars)
        adata.X = X_try
if _is_valid_2d(adata.X, n_obs_expected=n_cells):
    expr_source = ('X', None)  # (where, layer_name)
else:
    # Try layers that look like expression
    candidate_layers = []
    if hasattr(adata, 'layers'):
        for key in adata.layers.keys():
            lk = key.lower()
            if any(tag in lk for tag in ['count', 'expr', 'log', 'norm']):
                candidate_layers.append(key)
    for layer in candidate_layers:
        X_layer = adata.layers[layer]
        if X_layer is not None:
            if hasattr(X_layer, 'A'):
                X_layer = X_layer.A
            X_layer = np.asarray(X_layer)
            if X_layer.ndim == 1 and X_layer.shape[0] == n_cells * adata.n_vars:
                X_layer = X_layer.reshape(n_cells, adata.n_vars)
                adata.layers[layer] = X_layer
        if _is_valid_2d(adata.layers[layer], n_obs_expected=n_cells):
            expr_source = ('layers', layer)
            break

# Try adata.raw as fallback
if expr_source is None and getattr(adata, 'raw', None) is not None:
    X_raw = adata.raw.X
    if X_raw is not None:
        if hasattr(X_raw, 'A'):
            X_raw = X_raw.A
        X_raw = np.asarray(X_raw)
        if X_raw.ndim == 1 and X_raw.shape[0] == n_cells * adata.raw.n_vars:
            X_raw = X_raw.reshape(n_cells, adata.raw.n_vars)
            adata.raw.X = X_raw
    if _is_valid_2d(adata.raw.X, n_obs_expected=n_cells):
        expr_source = ('raw', None)

# As a final fallback, if everything failed but adata has n_obs * n_vars values in .X, reshape and use it
if expr_source is None:
    X_fallback = adata.X
    if X_fallback is not None:
        if hasattr(X_fallback, 'A'):
            X_fallback = X_fallback.A
        X_fallback = np.asarray(X_fallback)
        if X_fallback.ndim == 1 and X_fallback.shape[0] == n_cells * adata.n_vars:
            X_fallback = X_fallback.reshape(n_cells, adata.n_vars)
            adata.X = X_fallback
            if _is_valid_2d(adata.X, n_obs_expected=n_cells):
                expr_source = ('X', None)

if expr_source is None:
    raise ValueError("Could not find or construct a valid 2D expression matrix in adata.X, adata.layers, or adata.raw.")

print("Using expression source for TF/signaling scores:", expr_source)

# 3) Extract expression for vCMs and TF/signaling genes, preserving vc_genes order
if expr_source[0] == 'X':
    sub = adata[vc_mask, vc_genes].copy()
    X = sub.X
elif expr_source[0] == 'layers':
    layer = expr_source[1]
    sub = adata[vc_mask, vc_genes].copy()
    X = sub.layers[layer]
else:  # 'raw'
    raw = adata.raw[vc_mask, :]
    # Map vc_genes to indices in raw.var_names to preserve order
    var_to_idx = {g: i for i, g in enumerate(list(raw.var_names))}
    gene_idx = [var_to_idx[g] for g in vc_genes if g in var_to_idx]
    if len(gene_idx) == 0:
        raise ValueError("None of the curated TF/signaling genes are present in adata.raw.var_names.")
    X = raw.X[:, gene_idx]

# Convert to dense and ensure 2D
if hasattr(X, 'A'):
    X = X.A
X = np.asarray(X)
if X.ndim != 2:
    raise ValueError(f"Expression matrix for vCM TF/signaling genes is not 2D (ndim={X.ndim}).")

n_vc, n_genes = X.shape
print(f"Expression matrix for vCMs and TF/signaling genes: {n_vc} cells x {n_genes} genes")

# 4) Compute per-gene z-scores across vCMs and aggregate per cell
means = X.mean(axis=0)
stds = X.std(axis=0, ddof=1)
zero_var_genes = np.sum(stds == 0)
if zero_var_genes > 0:
    print(f"Warning: {zero_var_genes} TF/signaling genes have zero variance across vCMs and will be treated as constant in the score.")
stds_safe = np.where(stds == 0, 1.0, stds)
X_z = (X - means) / stds_safe

# Average z-scores across genes per cell
vc_scores = X_z.mean(axis=1)

# 5) Store scores in adata.obs for all cells (NaN for non-vCMs)
score_name = 'vCM_TF_signaling_score'
full_scores = np.full(adata.n_obs, np.nan, dtype=float)
full_scores[vc_mask.values] = vc_scores
adata.obs[score_name] = full_scores

print(f"Stored per-cell TF/signaling score in adata.obs['{score_name}'] for vCMs; non-vCMs set to NaN.")

# Optional QC summary by vCM subtype
vc_scores_series = pd.Series(vc_scores, index=adata.obs.index[vc_mask])
qc_summary = vc_scores_series.groupby(adata.obs.loc[vc_mask, 'Populations']).describe()
print("\nTF/signaling score summary by vCM Population:")
print(qc_summary)


Total ventricular CM cells: 100637
vCM Populations: ['vCM-His-Purkinje', 'vCM-LV-AV', 'vCM-LV-Compact', 'vCM-LV-Trabecular', 'vCM-Proliferating', 'vCM-RV-AV', 'vCM-RV-Compact', 'vCM-RV-Trabecular']


ValueError: Could not find or construct a valid 2D expression matrix in adata.X, adata.layers, or adata.raw.

### Agent Interpretation

Current analysis step failed to run. Try an alternative approach

## Next Steps
Step 1: Verify ventricular cardiomyocyte TF/signaling scores and compute spatial kNN graphs per Sample_ID on obsm['spatial'] restricted to vCMs, derive neighborhood-mean TF/signaling scores, and quantify spatial coherence via cell vs neighborhood score correlations and simple regressions.
Step 2: Within ventricular cardiomyocytes, model associations between TF/signaling scores (per-cell and neighborhood-mean) and Purity using Pearson/Spearman correlations and linear regression stratified by Sample_ID and vCM subtype, explicitly reporting effect sizes and p-values.
Step 3: Using only ventricular cardiomyocytes and their TF/signaling gene expression, perform PCA, construct a kNN graph, run Leiden clustering to define TF-based maturation states, and quantitatively compare these TF-defined states to existing Leiden clusters and Populations via contingency tables, chi-squared tests, and Purity distribution tests (Kruskal–Wallis and pairwise Mann–Whitney), emphasizing TF states that cut across existing labels.

## This code restricts the analysis to ventricular cardiomyocytes, builds spatial k-nearest neighbor graphs within each sample, and computes per-cell neighborhood-mean vCM TF/signaling scores to assess how spatially coherent those scores are. It then quantifies this spatial coherence by correlating each cell’s score with its neighborhood mean (overall and within vCM subpopulations), providing sample- and population-level statistics on spatial organization of the signaling program.

In [ ]:

import numpy as np
import pandas as pd
from scipy import stats
import scanpy as sc
from scipy.spatial import cKDTree

# --- Step 1: Verify vCM TF/signaling scores, build spatial kNN per Sample_ID, and quantify spatial coherence ---

# 0) Basic checks for required metadata and TF/signaling score
required_obs = ['Populations', 'Purity']

for col in required_obs:
    if col not in adata.obs.columns:
        raise ValueError(f"Expected '{col}' in adata.obs but did not find it.")

# Handle missing vCM_TF_signaling_score by creating a placeholder column of NaNs
if 'vCM_TF_signaling_score' not in adata.obs.columns:
    adata.obs['vCM_TF_signaling_score'] = np.nan

if 'spatial' not in adata.obsm.keys():
    raise ValueError("Expected spatial coordinates in adata.obsm['spatial'] but did not find them.")

# Identify ventricular cardiomyocytes (vCMs) from Populations
pop = adata.obs['Populations'].astype(str)
vc_labels = [p for p in pop.unique().tolist() if isinstance(p, str) and p.upper().startswith('VCM')]
vc_mask = pop.isin(vc_labels)
if vc_mask.sum() == 0:
    raise ValueError("No ventricular cardiomyocytes found based on Populations starting with 'vCM' (case-insensitive).")

print(f"Total ventricular CM cells: {int(vc_mask.sum())}")
print("vCM Populations:", sorted(set(pop[vc_mask].tolist())))

# Ensure Sample_ID exists; if not, create a single-sample label
if 'Sample_ID' not in adata.obs.columns:
    adata.obs['Sample_ID'] = 'Sample_1'

# Quick QC of TF/signaling score distribution in vCMs
vcm_scores = adata.obs.loc[vc_mask, 'vCM_TF_signaling_score']
print("\nTF/signaling score summary for all vCMs:")
print(vcm_scores.describe())

sample_ids = adata.obs.loc[vc_mask, 'Sample_ID'].unique().tolist()
for sid in sample_ids:
    sid_mask = (adata.obs['Sample_ID'] == sid) & vc_mask
    print(f"\nTF/signaling score summary for vCMs in Sample_ID={sid}:")
    print(adata.obs.loc[sid_mask, 'vCM_TF_signaling_score'].describe())

# Prepare containers for neighborhood-mean scores
nb_mean_col = 'vCM_TF_signaling_score_nb_mean'
nb_degree_col = 'vCM_spatial_kNN_degree'
adata.obs[nb_mean_col] = np.nan
adata.obs[nb_degree_col] = np.nan

# Parameters for spatial kNN
default_k = 15

# Function to compute neighborhood-mean scores for one sample
small_eps = 1e-8

def compute_nb_means_for_sample(sample_id, k=default_k):
    sample_mask = (adata.obs['Sample_ID'] == sample_id) & vc_mask
    n_cells = int(sample_mask.sum())
    if n_cells < 2:
        # No meaningful neighborhood structure with <2 cells
        adata.obs.loc[sample_mask, nb_mean_col] = np.nan
        adata.obs.loc[sample_mask, nb_degree_col] = 0
        print(f"  Sample_ID={sample_id}: only {n_cells} vCM cell(s); skipping kNN.")
        return

    coords = adata.obsm['spatial'][sample_mask.values, :]
    scores = adata.obs.loc[sample_mask, 'vCM_TF_signaling_score'].values

    # Build k-d tree on spatial coordinates
    tree = cKDTree(coords)

    # Use min(k, n_cells-1) neighbors (excluding the cell itself)
    k_eff = min(k, n_cells - 1)

    # Query k_eff+1 neighbors including self, then drop self
    dists, idxs = tree.query(coords, k=k_eff + 1)
    # For both k_eff==1 and k_eff>1, first column is self
    nb_indices = idxs[:, 1:]

    # Compute neighborhood-mean TF/signaling score per cell
    nb_scores = []
    nb_counts = []
    for i in range(n_cells):
        neigh_idx = nb_indices[i]
        # Filter out any negative indices (should not occur but safe)
        neigh_idx = neigh_idx[neigh_idx >= 0]
        if neigh_idx.size == 0:
            nb_scores.append(np.nan)
            nb_counts.append(0)
        else:
            nb_scores.append(float(np.nanmean(scores[neigh_idx])))
            nb_counts.append(int(neigh_idx.size))

    adata.obs.loc[sample_mask, nb_mean_col] = nb_scores
    adata.obs.loc[sample_mask, nb_degree_col] = nb_counts

# 1) Compute neighborhood-mean TF/signaling scores per Sample_ID for vCMs
print("\nComputing spatial kNN neighborhoods and neighborhood-mean TF/signaling scores per Sample_ID for vCMs...")
for sid in sample_ids:
    print(f"  Processing Sample_ID = {sid}")
    compute_nb_means_for_sample(sid, k=default_k)

# Check how many vCMs have valid neighborhood means
valid_nb = vc_mask & adata.obs[nb_mean_col].notnull()
print(f"\nvCMs with valid neighborhood-mean scores: {int(valid_nb.sum())} / {int(vc_mask.sum())}")

# 2) Quantify spatial coherence: correlation between cell scores and neighborhood-mean scores
print("\nSpatial coherence of vCM TF/signaling scores (cell vs. neighborhood-mean):")

results = []
for sid in sample_ids:
    sid_mask = (adata.obs['Sample_ID'] == sid) & valid_nb
    if sid_mask.sum() < 10:
        print(f"  Sample_ID={sid}: only {int(sid_mask.sum())} vCMs with valid neighborhood scores; skipping coherence stats.")
        continue
    x = adata.obs.loc[sid_mask, 'vCM_TF_signaling_score'].values
    y = adata.obs.loc[sid_mask, nb_mean_col].values

    # Guard against constant or near-constant vectors for correlations/regression
    if (np.nanstd(x) < small_eps) or (np.nanstd(y) < small_eps):
        print(f"  Sample_ID={sid}: TF/signaling scores or neighborhood means nearly constant; skipping coherence stats.")
        continue

    # Pearson and Spearman correlations
    pearson_r, pearson_p = stats.pearsonr(x, y)
    spearman_r, spearman_p = stats.spearmanr(x, y)

    # Simple linear regression of neighborhood-mean on cell score
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

    n_cells_sample = int(sid_mask.sum())
    results.append({
        'Sample_ID': sid,
        'n_vCM_cells': n_cells_sample,
        'pearson_r': pearson_r,
        'pearson_p': pearson_p,
        'spearman_r': spearman_r,
        'spearman_p': spearman_p,
        'linreg_slope': slope,
        'linreg_intercept': intercept,
        'linreg_r2': r_value ** 2,
        'linreg_p': p_value,
        'linreg_std_err': std_err
    })

# Convert to DataFrame for a clean text table
if len(results) > 0:
    res_df = pd.DataFrame(results)
    # Order columns
    res_df = res_df[[
        'Sample_ID', 'n_vCM_cells',
        'pearson_r', 'pearson_p',
        'spearman_r', 'spearman_p',
        'linreg_slope', 'linreg_intercept', 'linreg_r2', 'linreg_p', 'linreg_std_err'
    ]]
    print(res_df.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
else:
    print("No samples with sufficient vCM cells and valid neighborhood-mean scores to compute spatial coherence.")

# 3) Optional: spatial coherence stratified by vCM Population within each Sample_ID
print("\nSpatial coherence stratified by vCM Population and Sample_ID:")

stratified_rows = []
for sid in sample_ids:
    sid_mask = (adata.obs['Sample_ID'] == sid) & valid_nb
    if sid_mask.sum() < 10:
        continue
    pops_in_sid = adata.obs.loc[sid_mask, 'Populations'].unique().tolist()
    for p_lab in pops_in_sid:
        sub_mask = sid_mask & (adata.obs['Populations'] == p_lab)
        if sub_mask.sum() < 20:
            # Require at least 20 cells for a stable estimate
            continue
        x = adata.obs.loc[sub_mask, 'vCM_TF_signaling_score'].values
        y = adata.obs.loc[sub_mask, nb_mean_col].values

        # Guard against constant or near-constant vectors
        if (np.nanstd(x) < small_eps) or (np.nanstd(y) < small_eps):
            continue

        pearson_r, pearson_p = stats.pearsonr(x, y)
        spearman_r, spearman_p = stats.spearmanr(x, y)
        slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

        stratified_rows.append({
            'Sample_ID': sid,
            'vCM_Population': p_lab,
            'n_cells': int(sub_mask.sum()),
            'pearson_r': pearson_r,
            'pearson_p': pearson_p,
            'spearman_r': spearman_r,
            'spearman_p': spearman_p,
            'linreg_slope': slope,
            'linreg_intercept': intercept,
            'linreg_r2': r_value ** 2,
            'linreg_p': p_value,
            'linreg_std_err': std_err
        })

if len(stratified_rows) > 0:
    strat_df = pd.DataFrame(stratified_rows)
    strat_df = strat_df.sort_values(['Sample_ID', 'vCM_Population'])
    strat_df = strat_df[[
        'Sample_ID', 'vCM_Population', 'n_cells',
        'pearson_r', 'pearson_p',
        'spearman_r', 'spearman_p',
        'linreg_slope', 'linreg_intercept', 'linreg_r2', 'linreg_p', 'linreg_std_err'
    ]]
    print(strat_df.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
else:
    print("No vCM Population × Sample_ID combinations with ≥20 cells and valid neighborhood-mean scores for stratified spatial coherence analysis.")


Total ventricular CM cells: 100637
vCM Populations: ['vCM-His-Purkinje', 'vCM-LV-AV', 'vCM-LV-Compact', 'vCM-LV-Trabecular', 'vCM-Proliferating', 'vCM-RV-AV', 'vCM-RV-Compact', 'vCM-RV-Trabecular']

TF/signaling score summary for all vCMs:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: vCM_TF_signaling_score, dtype: float64

TF/signaling score summary for vCMs in Sample_ID=R77_4C4:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: vCM_TF_signaling_score, dtype: float64

TF/signaling score summary for vCMs in Sample_ID=R78_4C12:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: vCM_TF_signaling_score, dtype: float64

TF/signaling score summary for vCMs in Sample_ID=R78_4C15:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: vCM_TF_signaling_score, 

  Processing Sample_ID = R78_4C12


  Processing Sample_ID = R78_4C15



vCMs with valid neighborhood-mean scores: 0 / 100637

Spatial coherence of vCM TF/signaling scores (cell vs. neighborhood-mean):
  Sample_ID=R77_4C4: only 0 vCMs with valid neighborhood scores; skipping coherence stats.
  Sample_ID=R78_4C12: only 0 vCMs with valid neighborhood scores; skipping coherence stats.
  Sample_ID=R78_4C15: only 0 vCMs with valid neighborhood scores; skipping coherence stats.
No samples with sufficient vCM cells and valid neighborhood-mean scores to compute spatial coherence.

Spatial coherence stratified by vCM Population and Sample_ID:
No vCM Population × Sample_ID combinations with ≥20 cells and valid neighborhood-mean scores for stratified spatial coherence analysis.


### Agent Interpretation

All ventricular cardiomyocytes were found and the spatial kNN graph per sample was successfully built, but the core quantity for the hypothesis — `vCM_TF_signaling_score` — is entirely missing (all NaN). As a result:

- Neighborhood means are also NaN.
- No spatial coherence statistics (cell vs neighborhood-mean correlations or regressions) can be computed.
- Nothing can yet be said about spatially coherent maturation “domains” or their relationship to Purity, or about whether these patterns are independent of existing Leiden clusters/Populations.

So at this stage, the hypothesis is entirely untested rather than refuted.

Concrete recommendations for next steps:

1. **Define / recover the TF/signaling score before proceeding.**
   - The code currently just creates a NaN placeholder if `vCM_TF_signaling_score` is missing. That’s appropriate for safety, but you now need to:
     - Either load the pre-computed score from disk (if it exists in a previous object or file), or
     - Compute it de novo from the current AnnData.
   - If computing de novo, do it in a clearly separate step, then re-run this spatial coherence analysis:
     - Decide on the TF/signaling program: for example, the subset of panel genes that are transcription factors / signaling components, potentially separated into “maturation-promoting” vs “immature” programs if that fits your hypothesis.
     - Compute per-cell scores using something like:
       - Simple average / z-scored mean expression over the selected genes.
       - Or Scanpy’s `tl.score_genes` for a more standard approach.
     - Restrict to vCMs (as you already do) or compute globally and then subset.

2. **Sanity-check the new TF/signaling score distribution.**
   Once the score is non-NaN, repeat the initial QC but verify:
   - Non-zero `count` for vCMs and per Sample_ID.
   - Reasonable spread (non-trivial standard deviation; no extreme outliers from technical artifacts).

3. **Re-run this same spatial kNN / neighborhood-mean step.**
   - With valid scores, the existing code should:
     - Fill `vCM_TF_signaling_score_nb_mean` and `vCM_spatial_kNN_degree`.
     - Produce per-sample Pearson/Spearman correlations and linear regression fits.
   - Interpret the spatial coherence:
     - Strong positive correlations (e.g. r > 0.5 with small p-values) would support that TF/signaling maturation scores form spatially coherent domains.
     - Compare coherence strength across samples and across vCM Populations (His–Purkinje vs compact vs trabecular, etc.) using the stratified section.

4. **Consider robustness checks once scores exist.**
   - Vary spatial neighborhood size:
     - Repeat with different k (e.g. 5, 10, 20, 30) to ensure that coherence is not an artifact of a single k choice.
   - Assess whether coherence is preserved when you:
     - Randomly permute scores within each sample (negative control), to show that observed coherence exceeds random expectation.
     - Optionally, use distance-binned correlations (near vs far neighbors) for a more explicit notion of spatial range.

5. **Prepare for the next steps of your stated plan (once scores are in place).**
   - Step 2 (Purity association):
     - Use both per-cell and neighborhood-mean TF/signaling scores to correlate with Purity, per Sample_ID and per vCM subtype.
     - Explicitly store effect sizes and p-values in tables, as you’ve done here.
   - Step 3 (TF-based maturation states):
     - Use the underlying TF/signaling gene expression matrix for vCMs to build PCA, kNN, and Leiden clusters representing TF-defined maturation states.
     - Later, compare those clusters vs existing Populations and Purity distributions.

Summary: the current step is structurally correct but cannot inform the hypothesis because the key TF/signaling score is entirely missing. The immediate priority is to reconstruct or load a biologically meaningful `vCM_TF_signaling_score` and then re-run this spatial neighborhood analysis; only then can you test whether TF-based maturation scores form spatially coherent patterns independent of existing annotations and associated with Purity.

## Next Steps
Step 1: Recompute a robust per-cell ventricular cardiomyocyte TF/signaling score directly from a clearly identified 2D expression matrix (preferring adata.X, then expression-like layers, then adata.raw), restricted to adata.uns['vc_tf_signaling_genes'], store it in adata.obs['vCM_TF_signaling_score'], and provide concise QC summaries to ensure it is non-NaN with reasonable variance across vCMs and samples.
Step 2: Using the recomputed vCM TF/signaling scores, rebuild spatial kNN neighborhoods per Sample_ID on obsm['spatial'] restricted to vCMs, compute neighborhood-mean scores, and then quantify (i) spatial coherence (cell vs neighborhood mean correlations and regressions) and (ii) association with Purity (correlations and regressions) per sample and major vCM subtype, printing effect sizes and p-values.

## This code robustly recomputes per-cell ventricular cardiomyocyte TF/signaling scores by selecting a valid 2D expression matrix (preferring X, then expression-like layers, then raw), extracting curated TF/signaling genes for vCMs, computing z-score-based gene averages, and writing the resulting scores into adata.obs with basic QC summaries.

In [ ]:

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import stats

# --- Step 2 (redefined as new first step here): robustly recompute vCM TF/signaling scores ---

# 0) Basic checks
if 'Populations' not in adata.obs.columns:
    raise ValueError("Expected 'Populations' in adata.obs.")
if 'vc_tf_signaling_genes' not in adata.uns:
    raise ValueError("Expected curated TF/signaling gene list in adata.uns['vc_tf_signaling_genes'].")

vc_genes_all = list(adata.uns['vc_tf_signaling_genes'])
if len(vc_genes_all) == 0:
    raise ValueError("adata.uns['vc_tf_signaling_genes'] is empty; cannot compute TF/signaling scores.")

# Identify ventricular cardiomyocytes (vCMs)
pop = adata.obs['Populations'].astype(str)
vc_labels = [p for p in pop.unique().tolist() if isinstance(p, str) and p.upper().startswith('VCM')]
vc_mask = pop.isin(vc_labels)
if vc_mask.sum() == 0:
    raise ValueError("No ventricular cardiomyocytes found based on Populations starting with 'vCM' (case-insensitive).")

print(f"Total ventricular CM cells: {int(vc_mask.sum())}")
print("vCM Populations:", sorted(set(pop[vc_mask].tolist())))

# 1) Choose a valid 2D expression source (X, expression-like layers, or raw)

def _is_valid_2d(X, n_obs_expected=None, n_vars_expected=None):
    if X is None:
        return False
    if hasattr(X, 'A'):
        X = X.A
    X = np.asarray(X)
    if X.ndim != 2:
        return False
    if (n_obs_expected is not None) and (X.shape[0] != n_obs_expected):
        return False
    if (n_vars_expected is not None) and (X.shape[1] != n_vars_expected):
        return False
    return True

n_cells = adata.n_obs
n_genes_total = adata.n_vars
expr_source = None  # ('X' / 'layers' / 'raw', layer_name or None)

# Try adata.X first
X_try = adata.X
if X_try is not None:
    if hasattr(X_try, 'A'):
        X_try = X_try.A
    X_try = np.asarray(X_try)
    if X_try.ndim == 1 and X_try.size == n_cells * n_genes_total:
        X_try = X_try.reshape(n_cells, n_genes_total)
        adata.X = X_try
if _is_valid_2d(adata.X, n_obs_expected=n_cells, n_vars_expected=n_genes_total):
    expr_source = ('X', None)

# If X not usable, try expression-like layers
if expr_source is None and hasattr(adata, 'layers'):
    candidate_layers = []
    for key in adata.layers.keys():
        lk = key.lower()
        if any(tag in lk for tag in ['count', 'expr', 'log', 'norm']):
            candidate_layers.append(key)
    for layer in candidate_layers:
        X_layer = adata.layers[layer]
        if X_layer is not None:
            if hasattr(X_layer, 'A'):
                X_layer = X_layer.A
            X_layer = np.asarray(X_layer)
            if X_layer.ndim == 1 and X_layer.size == n_cells * n_genes_total:
                X_layer = X_layer.reshape(n_cells, n_genes_total)
                adata.layers[layer] = X_layer
        if _is_valid_2d(adata.layers[layer], n_obs_expected=n_cells, n_vars_expected=n_genes_total):
            expr_source = ('layers', layer)
            break

# If still nothing, try adata.raw
if expr_source is None and getattr(adata, 'raw', None) is not None:
    X_raw = adata.raw.X
    n_raw_genes = adata.raw.n_vars
    if X_raw is not None:
        if hasattr(X_raw, 'A'):
            X_raw = X_raw.A
        X_raw = np.asarray(X_raw)
        if X_raw.ndim == 1 and X_raw.size == n_cells * n_raw_genes:
            X_raw = X_raw.reshape(n_cells, n_raw_genes)
    if _is_valid_2d(X_raw, n_obs_expected=n_cells, n_vars_expected=n_raw_genes):
        expr_source = ('raw', None)

# As a last resort, fall back to any 2D matrix even if sizes mismatch expectations
if expr_source is None:
    if adata.X is not None:
        X_tmp = adata.X
        if hasattr(X_tmp, 'A'):
            X_tmp = X_tmp.A
        X_tmp = np.asarray(X_tmp)
        if X_tmp.ndim == 1:
            side = int(np.sqrt(X_tmp.size))
            if side * side == X_tmp.size:
                adata.X = X_tmp.reshape(side, side)
                expr_source = ('X', None)
        elif X_tmp.ndim == 2:
            expr_source = ('X', None)
    if expr_source is None and getattr(adata, 'raw', None) is not None:
        X_raw = adata.raw.X
        if hasattr(X_raw, 'A'):
            X_raw = X_raw.A
        X_raw = np.asarray(X_raw)
        if X_raw.ndim == 2:
            expr_source = ('raw', None)

if expr_source is None:
    raise ValueError("Could not identify a valid 2D expression matrix in adata.X, expression-like layers, or adata.raw.")

print("Using expression source for TF/signaling scoring:", expr_source)

# 2) Extract expression for vCMs and TF/signaling genes in the chosen source
if expr_source[0] in ['X', 'layers']:
    # Restrict gene list to those present in adata.var_names
    vc_genes = [g for g in vc_genes_all if g in adata.var_names]
    if len(vc_genes) == 0:
        raise ValueError("None of the curated TF/signaling genes are present in adata.var_names.")
    sub = adata[vc_mask, vc_genes].copy()
    if expr_source[0] == 'X':
        X = sub.X
    else:
        layer = expr_source[1]
        X = sub.layers[layer]
else:  # 'raw'
    raw = adata.raw[vc_mask, :]
    var_to_idx = {g: i for i, g in enumerate(list(raw.var_names))}
    gene_idx = [var_to_idx[g] for g in vc_genes_all if g in var_to_idx]
    if len(gene_idx) == 0:
        raise ValueError("None of the curated TF/signaling genes are present in adata.raw.var_names.")
    X = raw.X[:, gene_idx]

# Ensure dense 2D
if hasattr(X, 'A'):
    X = X.A
X = np.asarray(X)
if X.ndim != 2:
    raise ValueError(f"Expression matrix for vCM TF/signaling genes is not 2D (ndim={X.ndim}).")

n_vc, n_tf = X.shape
print(f"Expression matrix for vCM TF/signaling genes: {n_vc} cells x {n_tf} genes")

# 3) Compute per-gene z-scores across vCMs and aggregate into a single score per cell
means = X.mean(axis=0)
stds = X.std(axis=0, ddof=1)
zero_var = np.sum(stds == 0)
if zero_var > 0:
    print(f"Warning: {zero_var} TF/signaling genes have zero variance across vCMs; treating them as constant in scoring.")
stds_safe = np.where(stds == 0, 1.0, stds)
X_z = (X - means) / stds_safe

vc_scores = X_z.mean(axis=1)

# 4) Write scores back to adata.obs for all cells (NaN for non-vCMs)
score_name = 'vCM_TF_signaling_score'
full_scores = np.full(adata.n_obs, np.nan, dtype=float)
full_scores[vc_mask.to_numpy()] = vc_scores
adata.obs[score_name] = full_scores

print(f"Stored recomputed TF/signaling scores in adata.obs['{score_name}'] for vCMs; non-vCMs set to NaN.")

# 5) Concise QC: distribution overall and by Sample_ID
print("\nTF/signaling score summary for vCMs (all samples):")
print(pd.Series(vc_scores).describe())

if 'Sample_ID' in adata.obs.columns:
    print("\nTF/signaling score summary for vCMs by Sample_ID:")
    vc_obs = adata.obs.loc[vc_mask, ['Sample_ID', score_name]]
    print(vc_obs.groupby('Sample_ID')[score_name].describe())
else:
    print("\nNo 'Sample_ID' column found; skipping per-sample vCM TF/signaling score summary.")


Total ventricular CM cells: 100637
vCM Populations: ['vCM-His-Purkinje', 'vCM-LV-AV', 'vCM-LV-Compact', 'vCM-LV-Trabecular', 'vCM-Proliferating', 'vCM-RV-AV', 'vCM-RV-Compact', 'vCM-RV-Trabecular']


ValueError: Could not identify a valid 2D expression matrix in adata.X, expression-like layers, or adata.raw.

### Agent Interpretation

Current analysis step failed to run. Try an alternative approach

## Next Steps
Step 1: Since the recomputation of vCM TF/signaling scores still failed (ending with an error message and leaving the spatial/Purity hypotheses untested), implement an alternative, minimal but robust per-cell scoring strategy that completely bypasses the fragile automatic source-detection logic: explicitly pull expression for adata.uns['vc_tf_signaling_genes'] from adata.X only (assuming it is already a valid 2D matrix as per the dataset summary), restrict to ventricular cardiomyocytes, compute a simple z-score-based mean TF/signaling score per vCM, store it in adata.obs['vCM_TF_signaling_score'], and print concise QC summaries (non-NaN counts, mean, standard deviation, per-sample summaries, and simple sanity checks on variance and fraction of valid scores) to confirm that the scores are usable for downstream spatial and Purity analyses.

## This code recomputes a per-cell TF/signaling score for ventricular cardiomyocytes by assuming adata.X is a valid 2D expression matrix, extracting curated TF/signaling genes, z-scoring them across vCMs, averaging per cell, storing the result in adata.obs, and printing concise QC and sanity-check summaries to ensure the scores are non-NaN and variable enough for downstream spatial and Purity analyses.

In [ ]:

import numpy as np
import pandas as pd

# --- Robust, minimal recomputation of vCM TF/signaling scores using only adata.X ---

# 0) Basic checks for required structures and explicit assumptions
if 'Populations' not in adata.obs.columns:
    raise ValueError("Expected 'Populations' in adata.obs but did not find it.")
if 'vc_tf_signaling_genes' not in adata.uns:
    raise ValueError("Expected curated TF/signaling gene list in adata.uns['vc_tf_signaling_genes'].")

vc_genes_all = list(adata.uns['vc_tf_signaling_genes'])
if len(vc_genes_all) == 0:
    raise ValueError("adata.uns['vc_tf_signaling_genes'] is empty; cannot compute TF/signaling scores.")

# 1) Identify ventricular cardiomyocytes (vCMs)
pop = adata.obs['Populations'].astype(str)
vc_labels = [p for p in pop.unique().tolist() if isinstance(p, str) and p.upper().startswith('VCM')]
vc_mask = pop.isin(vc_labels)
if vc_mask.sum() == 0:
    raise ValueError("No ventricular cardiomyocytes found based on Populations starting with 'vCM' (case-insensitive).")

print(f"Total ventricular CM cells: {int(vc_mask.sum())}")
print("vCM Populations:", sorted(set(pop[vc_mask].tolist())))

# 2) Assert that adata.X is a valid 2D matrix and use it directly
X_global = adata.X
if X_global is None:
    raise ValueError("adata.X is None; this minimal scorer requires a 2D expression matrix in adata.X.")

# Handle scalar/0-dim edge case before conversion
if np.ndim(X_global) == 0:
    raise ValueError(
        "adata.X appears to be scalar (ndim=0); it must be a 2D expression matrix with shape (n_cells, n_genes)."
    )

# Convert sparse -> dense if needed
if hasattr(X_global, 'A'):
    X_global = X_global.A
X_global = np.asarray(X_global)

if X_global.ndim != 2:
    raise ValueError(
        "adata.X must be a 2D expression matrix for this scorer; "
        f"got ndim={X_global.ndim}. Please fix the AnnData object upstream."
    )

n_cells, n_genes_total = X_global.shape
if n_cells != adata.n_obs:
    raise ValueError(f"adata.X first dimension ({n_cells}) does not match n_obs ({adata.n_obs}).")

# 3) Restrict TF/signaling genes to those present in adata.var_names
present_mask = [g in adata.var_names for g in vc_genes_all]
vc_genes = [g for g, keep in zip(vc_genes_all, present_mask) if keep]
if len(vc_genes) == 0:
    raise ValueError("None of the curated TF/signaling genes are present in adata.var_names.")

print(f"Number of TF/signaling genes found in var_names: {len(vc_genes)}")

# Get column indices for these genes
var_to_idx = {g: i for i, g in enumerate(list(adata.var_names))}
gene_idx = [var_to_idx[g] for g in vc_genes]

# 4) Extract expression matrix only for vCMs and these TF/signaling genes
vc_indices = np.where(vc_mask.to_numpy())[0]
X_vc_tf = X_global[np.ix_(vc_indices, gene_idx)]  # shape: (n_vc, n_tf)

if X_vc_tf.ndim != 2 or X_vc_tf.shape[0] == 0 or X_vc_tf.shape[1] == 0:
    raise ValueError(
        f"Unexpected shape for vCM TF/signaling expression submatrix: {X_vc_tf.shape}; "
        "cannot compute scores."
    )

n_vc, n_tf = X_vc_tf.shape
print(f"Expression submatrix for vCMs and TF/signaling genes: {n_vc} cells x {n_tf} genes")

# 5) Compute per-gene z-scores across vCMs and average per cell
means = X_vc_tf.mean(axis=0)
stds = X_vc_tf.std(axis=0, ddof=1)
zero_var = np.sum(stds == 0)
if zero_var > 0:
    print(f"Warning: {zero_var} TF/signaling genes have zero variance across vCMs; treating them as constant in scoring.")

stds_safe = np.where(stds == 0, 1.0, stds)
X_vc_tf_z = (X_vc_tf - means) / stds_safe

vc_scores = X_vc_tf_z.mean(axis=1)  # one score per vCM

# 6) Write scores back to adata.obs (vCMs get scores, others NaN)
score_name = 'vCM_TF_signaling_score'
full_scores = np.full(adata.n_obs, np.nan, dtype=float)
full_scores[vc_indices] = vc_scores
adata.obs[score_name] = full_scores

print(f"Stored vCM TF/signaling scores in adata.obs['{score_name}'] for {n_vc} vCMs; non-vCMs set to NaN.")

# 7) Concise QC summaries and sanity checks
vc_scores_series = pd.Series(vc_scores, index=adata.obs.index[vc_mask])

non_nan_frac = float(np.mean(~np.isnan(vc_scores)))
std_all = float(np.std(vc_scores, ddof=1)) if vc_scores.size > 1 else 0.0

print("\nTF/signaling score summary for vCMs (all samples):")
print(vc_scores_series.describe())
print(f"Fraction of vCMs with non-NaN scores: {non_nan_frac:.3f}")
print(f"Overall standard deviation of vCM scores: {std_all:.4f}")
if std_all < 1e-6:
    print("Warning: vCM TF/signaling scores are effectively constant (std < 1e-6); downstream spatial/Purity analyses may be uninformative.")

if 'Sample_ID' in adata.obs.columns:
    print("\nTF/signaling score summary for vCMs by Sample_ID:")
    vc_obs = adata.obs.loc[vc_mask, ['Sample_ID', score_name]]
    print(vc_obs.groupby('Sample_ID')[score_name].describe())
else:
    print("\nNo 'Sample_ID' column found; skipping per-sample score summary.")


Total ventricular CM cells: 100637
vCM Populations: ['vCM-His-Purkinje', 'vCM-LV-AV', 'vCM-LV-Compact', 'vCM-LV-Trabecular', 'vCM-Proliferating', 'vCM-RV-AV', 'vCM-RV-Compact', 'vCM-RV-Trabecular']


ValueError: adata.X must be a 2D expression matrix for this scorer; got ndim=0. Please fix the AnnData object upstream.

### Agent Interpretation

Current analysis step failed to run. Try an alternative approach